In [2]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [3]:
from utils import *
import os
import torch
import tensorflow as tf
from cnn.model import ConvBlock
import cnn_tf.model as tf_model
import numpy as np

In [4]:
# disable all tensorflow logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [5]:
# Load config file
file_path = os.path.join(os.getcwd(), 'config_files', 'config2.txt')
config_file = load_yaml(file_path)

## Test ConvBlock

In [6]:
kernel = 3
stride = 1
filters = 32
mu=0.9
epsilon=2e-5
conv_block = ConvBlock(kernel = kernel,
                       strides = stride, 
                       filters = filters, 
                       mu=mu,
                       epsilon=epsilon)

In [7]:
tf_conv_block = tf_model.ConvBlock(kernel = kernel,
                                   strides = stride,
                                   filters = filters,
                                   mu=mu, 
                                   epsilon=epsilon)

In [8]:
# create a tensor with numpy an convert it to pytorch tensor
X = np.random.rand(1, 6, 6, 3).astype(np.float32)
x_torch = torch.from_numpy(X)
x_tf = tf.convert_to_tensor(X, dtype=tf.float32)

In [9]:
np.isclose(x_torch, x_tf).all()

True

In [10]:
y_torch = conv_block(x_torch).detach().numpy()
y_tf = tf_conv_block(x_tf, name="conv_block")

print(f"torch: {y_torch.shape}, tensorflow: {y_tf.shape}")

torch: (1, 6, 6, 32), tensorflow: (1, 6, 6, 32)


d:\qnas_torch\cnn_tf\model.py:68: UserWarning: `tf.layers.conv2d` is deprecated and will be removed in a future version. Please Use `tf.keras.layers.Conv2D` instead.
  return tf.compat.v1.layers.conv2d(inputs=inputs,
d:\qnas_torch\cnn_tf\model.py:90: UserWarning: `tf.layers.batch_normalization` is deprecated and will be removed in a future version. Please use `tf.keras.layers.BatchNormalization` instead. In particular, `tf.control_dependencies(tf.GraphKeys.UPDATE_OPS)` should not be used (consult the `tf.keras.layers.BatchNormalization` documentation).
  return tf.compat.v1.layers.batch_normalization(inputs=inputs,


In [11]:
np.isclose(y_torch, y_tf).all()

False

## Test Max Pooling

In [12]:
from cnn.model import MaxPooling

In [13]:
# create a a image tensor to test the max pooling layer
max_pool = MaxPooling(kernel=3, strides=1)
tf_max_pool = tf_model.MaxPooling(kernel=3, strides=1)

y_tf = tf_max_pool(x_tf, name="max_pool")
y_torch = max_pool(x_torch).detach().numpy()

print(f"torch: {y_torch.shape}, tensorflow: {y_tf.shape}")

torch: (1, 4, 4, 3), tensorflow: (1, 4, 4, 3)


d:\qnas_torch\cnn_tf\model.py:275: UserWarning: `tf.layers.max_pooling2d` is deprecated and will be removed in a future version. Please use `tf.keras.layers.MaxPooling2D` instead.
  return tf.compat.v1.layers.max_pooling2d(inputs=inputs,


In [14]:
np.isclose(y_torch, y_tf).all()

True

## Test Avg Pooling

In [15]:
from cnn.model import AvgPooling

avg_pool = AvgPooling(kernel=3, strides=1)
y_torch = avg_pool(x_torch).detach().numpy()

tf_avg_pool = tf_model.AvgPooling(kernel=3, strides=1)
y_tf = tf_avg_pool(x_tf, name='avg_pool')

print(f"torch: {y_torch.shape}, tensorflow: {y_tf.shape}")

torch: (1, 4, 4, 3), tensorflow: (1, 4, 4, 3)


d:\qnas_torch\cnn_tf\model.py:311: UserWarning: `tf.layers.average_pooling2d` is deprecated and will be removed in a future version. Please use `tf.keras.layers.AveragePooling2D` instead.
  return tf.compat.v1.layers.average_pooling2d(inputs=inputs,


In [16]:
np.isclose(y_torch, y_tf).all()

True

## Test Fully Connected Layer

In [17]:
# create a tensor with numpy an convert it to pytorch tensor
X = np.random.rand(32, 6, 6, 3).astype(np.float32)
x_torch = torch.from_numpy(X)
x_tf = tf.convert_to_tensor(X, dtype=tf.float32)

In [18]:
from cnn.model import FullyConnected

# flatten the tensor
num_features = x_torch.shape[1] * x_torch.shape[2] * x_torch.shape[3]
units = 20

fc_torch = FullyConnected(inputs_features=num_features, units=units)
fc_tf = tf_model.FullyConnected(units=units)

x_tor = torch.reshape(x_torch, [-1, num_features])
x_tfl = tf.reshape(x_tf, [-1, num_features])

y_torch = fc_torch(x_tor).detach().numpy()
y_tf = fc_tf(x_tfl, name='fc')

print(f"torch: {y_torch.shape}, tensorflow: {y_tf.shape}")

torch: (32, 20), tensorflow: (32, 20)


d:\qnas_torch\cnn_tf\model.py:345: UserWarning: `tf.layers.dense` is deprecated and will be removed in a future version. Please use `tf.keras.layers.Dense` instead.
  tensor = tf.compat.v1.layers.dense(inputs=inputs,


In [19]:
np.isclose(y_torch, y_tf).all()

False

## Test Pad Feature

In [20]:
# create a tensor with numpy an convert it to pytorch tensor
X = np.random.rand(1, 6, 6, 8).astype(np.float32)
X1 = np.random.rand(1, 6, 6, 6).astype(np.float32)

x_torch = torch.from_numpy(X)
x_tf = tf.convert_to_tensor(X, dtype=tf.float32)

x_torch1 = torch.from_numpy(X1)
x_tf1 = tf.convert_to_tensor(X1, dtype=tf.float32)


In [21]:
from cnn.model import pad_features

torch_tensors = [x_torch, x_torch1]
tf_tensors = [x_tf, x_tf1]


torch_padded = pad_features(tensors=torch_tensors)
tf_padded = tf_model.pad_features(tensors=tf_tensors)

torch_padded[0].shape, torch_padded[1].shape, tf_padded[0].shape, tf_padded[1].shape

(torch.Size([1, 6, 6, 8]),
 torch.Size([1, 6, 6, 8]),
 TensorShape([1, 6, 6, 8]),
 TensorShape([1, 6, 6, 8]))

## Test ResidualV1

In [34]:
from cnn.model import ResidualV1

kernel = 3
stride = 1
filters = 32
mu=0.9
epsilon=2e-5
Res_v1 = ResidualV1(kernel = kernel,
                       strides = stride, 
                       filters = filters, 
                       mu=mu,
                       epsilon=epsilon)

In [35]:
tf_Res = tf_model.ResidualV1(kernel = kernel,
                       strides = stride, 
                       filters = filters, 
                       mu=mu,
                       epsilon=epsilon)

In [36]:
# create a tensor with numpy an convert it to pytorch tensor
X = np.random.rand(1, 5, 5, 3).astype(np.float32)
x_torch = torch.from_numpy(X)
x_tf = tf.convert_to_tensor(X, dtype=tf.float32)

In [37]:
np.isclose(x_torch, x_tf).all()

True

In [38]:
y_torch = Res_v1(x_torch).detach().numpy()
y_tf = tf_Res(x_tf, name="conv_block")

print(f"torch: {y_torch.shape}, tensorflow: {y_tf.shape}")

inputs.shape: torch.Size([1, 3, 5, 5])
tensor.shape Layer 1: torch.Size([1, 32, 5, 5])
tensor.shape Layer 2: torch.Size([1, 32, 5, 5])
inputs.shape Layer 2: torch.Size([1, 32, 5, 5])
tensor.shape Pad: torch.Size([1, 32, 5, 5])
torch: (1, 5, 5, 32), tensorflow: (1, 5, 5, 32)


In [28]:
np.isclose(y_torch, y_tf).all()

False